# Train Multinomial Naive Bayes sentiment (IMDB reviews CSV)

Expects a CSV under **`data/imdb-review/`** or **`data/imbd-review/`** with:
- **Column 1:** review text (header may be `review`, `text`, `sentence`, etc.)
- **Column 2:** sentiment as **`positive`** / **`negative`**

Outputs **`models/artifacts/sentiment_mnb.joblib`** for the Spark streaming job (`streaming/processors/sentiment_processor.py`).

Run Jupyter from the **repository root** (or adjust `ROOT` below).

In [ ]:
from pathlib import Path
import re

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Repository root (parent of notebooks/)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Accept common folder names (typo imbd-review vs imdb-review)
for name in ("imdb-review", "imbd-review"):
    d = ROOT / "data" / name
    if d.is_dir():
        DATA_DIR = d
        break
else:
    DATA_DIR = ROOT / "data" / "imdb-review"
OUTPUT_PATH = ROOT / "models" / "artifacts" / "sentiment_mnb.joblib"

print("ROOT:", ROOT.resolve())
print("DATA_DIR:", DATA_DIR.resolve())
print("OUTPUT:", OUTPUT_PATH.resolve())

## 1. Locate CSV
Uses the **first `*.csv`** found in `data/imdb-review/`. Put your IMDB export there.

In [ ]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        f"Missing folder: {DATA_DIR}\n"
        "Create data/imdb-review/ (or data/imbd-review/) and place your CSV there."
    )

csv_files = sorted(DATA_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files in {DATA_DIR}")

CSV_PATH = csv_files[0]
print("Using:", CSV_PATH.name)

## 2. Load raw data
Auto-detect text / sentiment columns: prefer names `review` & `sentiment`, else **first two columns**.

In [ ]:
raw = pd.read_csv(CSV_PATH)
raw.columns = [c.strip() for c in raw.columns]

text_col = None
sent_col = None
for name in ("review", "text", "sentence", "comment", "Review", "TEXT"):
    if name in raw.columns:
        text_col = name
        break
for name in ("sentiment", "label", "polarity", "Sentiment", "LABEL"):
    if name in raw.columns:
        sent_col = name
        break

if text_col is None or sent_col is None:
    if raw.shape[1] >= 2:
        text_col, sent_col = raw.columns[0], raw.columns[1]
        print(f"Using first two columns as text={text_col!r}, sentiment={sent_col!r}")
    else:
        raise ValueError(f"Could not infer columns. Found: {list(raw.columns)}")

df = raw[[text_col, sent_col]].rename(columns={text_col: "review", sent_col: "sentiment"})
df.head()

## 3. Preprocessing
- Strip / normalize whitespace  
- Map sentiment to **0 (negative)** / **1 (positive)**  
- Drop empty reviews and invalid labels  
- Optional: drop duplicate review text

In [ ]:
def clean_text(s: str) -> str:
    s = "" if pd.isna(s) else str(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def normalize_sentiment(x) -> int | None:
    if pd.isna(x):
        return None
    v = str(x).strip().lower()
    if v in ("positive", "pos", "1", "true"):
        return 1
    if v in ("negative", "neg", "0", "false"):
        return 0
    return None


prep = df.copy()
prep["review"] = prep["review"].map(clean_text)
prep["label"] = prep["sentiment"].map(normalize_sentiment)
prep = prep.dropna(subset=["label"])
prep = prep[prep["review"] != ""]
prep["label"] = prep["label"].astype(int)
prep = prep.drop_duplicates(subset=["review"], keep="first")

print("Rows after cleaning:", len(prep))
print(prep["label"].value_counts())
if prep["label"].nunique() < 2:
    raise ValueError("Need both positive and negative labels after cleaning.")

X = prep["review"].tolist()
y = prep["label"].values
prep.head()

## 4. Train / validation split & model
Same pipeline as **`scripts/train_sentiment_nb.py`**: `TfidfVectorizer` + `MultinomialNB`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                max_features=20000,
                ngram_range=(1, 2),
                min_df=1,
                strip_accents="unicode",
            ),
        ),
        ("clf", MultinomialNB(alpha=0.1)),
    ]
)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred, target_names=["negative", "positive"]))

## 5. Fit on full data & save joblib
Retrain on **all** cleaned rows for the artifact used in production streaming.

In [ ]:
pipeline.fit(X, y)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, OUTPUT_PATH)
print("Saved:", OUTPUT_PATH)
print("Classes:", list(pipeline.classes_))